# Ok-k-mer

Use python310 conda environment

## Imports

In [32]:
import os
# from random import randint
from random import sample
from random import sample, seed as set_seed, randint
import numpy as np
import pandas as pd
from collections import Counter
from ete3 import Tree
from itertools import combinations, product
from scipy.stats import binom, geom, spearmanr, pearsonr

pd.options.display.max_rows = 40

In [2]:
# import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mplpatches
%matplotlib inline

plt.rcParams['figure.dpi'] = 300
plt.rcParams['agg.path.chunksize'] = 10000

In [3]:
# import plotnine

from plotnine import \
    ggplot, aes, theme, themes, element_blank, element_line, element_text, mapping, \
    geom_point, geom_jitter, geom_line, geom_bar, geom_boxplot, geom_violin, geom_histogram, \
    ggtitle, xlab, ylab, geom_vline, geom_hline, geom_abline, geom_col, geom_tile, \
    scale_x_discrete, scale_y_discrete, scale_x_continuous, scale_y_continuous, \
    scale_color_discrete, scale_fill_manual, scale_fill_hue, scale_fill_gradient, scale_color_manual, scale_color_hue, \
    guides, after_stat, \
    xlim, ylim
from plotnine.options import figure_size
from mizani.formatters import percent_format

# plotnine theme
def theme_tufte_func(base_size=19, base_family="sans", rotate=True, figure_size=None):
    ''' Plotnine plotting theme. '''
    thm = themes.theme_bw(base_size=base_size, base_family = base_family) + \
        theme(
            legend_background = element_blank(),
            legend_key = element_blank(),
            panel_background = element_blank(),
            panel_border = element_blank(),
            strip_background = element_blank(),
            plot_background = element_blank(),
            panel_grid = element_blank(),
            axis_line = element_line(colour = "black", size = 1),
            axis_text_y = element_text(colour = "black")
        )

    if rotate:
        thm += theme(axis_text_x=element_text(rotation=45, hjust=1))
    if figure_size is not None:
        thm += theme(figure_size=figure_size)
    return thm


## Helper functions

In [4]:
def yield_file_lines(path):
    ''' Read file from path line-by-line. '''
    with open(path, 'r') as in_file:
        for line in in_file:
            yield line.strip()

In [5]:
def calc_corr(l1, l2, pearson=False, prnt=False):
    if pearson:
        corr, pval = pearsonr(l1, l2)
    else:
        corr, pval = spearmanr(l1, l2)
    if prnt:
        print("Correlation coefficient:", corr)
        print("p-value:", pval)
    return corr, pval

In [6]:
def dist_tree_calc(filename1):
    ''' Calculate tree distance. '''
    tree1= Tree(filename1,format=1)
    # print('Tree from', filename1, 'len:', len(tree1))
    dist_tree={}
    samples = [i.name for i in tree1.get_leaves()]
    num_sample = len(samples)
    for i in range(num_sample):
        tax_i = samples[i]
        for j in range(i+1,num_sample):
            tax_j = samples[j]
            dist_tree[(min(tax_i, tax_j)+".fa", max(tax_i, tax_j)+".fa")] = tree1.get_distance(tax_i, tax_j, topology_only=False)
    # print('num samples:', num_sample)
    # print('xxx:', num_sample * (num_sample-1)/2)
    # print('tree len:', len(dist_tree))
    return dist_tree


## Make tree distance dict

In [10]:

def make_tree_dict(tree_path):
    """
    Given a path to a Newick tree file, return a symmetric dictionary of distances between all pairs of taxa names
    (with or without '.fa' extension).

    Parameters:
        tree_path (str): Path to the Newick tree file.

    Returns:
        dict: A dictionary of dictionaries: { taxon_name: { other_taxon: distance, ... }, ... }
    """
    dist_tree = dist_tree_calc(tree_path)

    names = set()
    processed_pairs = []
    for (g1, g2), dist in dist_tree.items():
        g1_clean = g1.rstrip('.fa')
        g2_clean = g2.rstrip('.fa')
        names.add(g1_clean)
        names.add(g2_clean)
        processed_pairs.append((g1_clean, g2_clean, float(dist)))

    tree_dict = {name: {other: 0.0 for other in names} for name in names}
    for g1, g2, d in processed_pairs:
        tree_dict[g1][g2] = d
        tree_dict[g2][g1] = d  # Ensure symmetry

    return tree_dict


In [11]:
cyano_tree_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/cyanobacteriales/o__Cyanobacteriales.nwk'
chlamydia_tree_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/chlamydiales/o__Chlamydiales.nwk'

In [42]:
cyano_nwk_dict = make_tree_dict(cyano_tree_path)
chlam_nwk_dict = make_tree_dict(chlamydia_tree_path)

cyano_nwk_dict
chlam_nwk_dict

{'GCF_017472245.1': {'GCF_017472245.1': 0.0,
  'GCA_025451995.1': 1.373699,
  'GCA_016213235.1': 1.4677516,
  'GCA_013288005.1': 1.4538411000000004,
  'GCA_020051235.1': 1.3897456,
  'GCA_016931275.1': 1.4702342,
  'GCA_011064905.1': 1.5325681,
  'GCA_011065235.1': 1.4340226000000003,
  'GCA_011064875.1': 1.5099650999999998,
  'GCA_903864395.1': 1.4701294000000003,
  'GCA_945952445.1': 1.4075865,
  'GCA_019694475.1': 1.4668358000000004,
  'GCA_018239815.1': 1.196839,
  'GCA_021372185.1': 1.4530326,
  'GCA_945957185.1': 1.3480543,
  'GCA_030427725.1': 1.4614853,
  'GCA_013288825.1': 1.3433121,
  'GCA_030430655.1': 1.598909,
  'GCF_015356815.2': 1.5805921,
  'GCA_027365315.1': 1.4657577000000002,
  'GCF_019453995.1': 1.57882552,
  'GCA_030044075.1': 1.3656703000000003,
  'GCA_020054065.1': 1.2651021,
  'GCA_013286595.1': 1.4409963000000001,
  'GCA_003151315.1': 1.488029,
  'GCF_000006685.1': 0.1062504,
  'GCA_903970255.1': 1.4522420000000003,
  'GCA_021772165.1': 1.5632633,
  'GCA_027348

In [43]:
cyano_sample_names = sorted(cyano_nwk_dict.keys())
chlam_sample_names = sorted(chlam_nwk_dict.keys())
len(cyano_sample_names), len(chlam_sample_names)

(585, 288)

In [31]:

def get_random_samples(sample_names, sample_fraction=0.1, seed=None):
    if seed is not None:
        set_seed(seed)
    num_samples = max(1, int(len(sample_names) * sample_fraction))
    random_samples = sample(sample_names, num_samples)
    return random_samples

# Example usage:
cyano_random_samples = get_random_samples(cyano_sample_names, sample_fraction=0.1, seed=42)
len(cyano_random_samples)

58

In [35]:
def parse_dashing_dfs(dir_path):
    """
    Parses all distance_* files in the given directory into a dictionary of pandas DataFrames.
    Each DataFrame skips the first line, assumes tab-separation and adjusts indices and columns as expected.

    Args:
        dir_path (str): Path to the directory containing distance_* files.

    Returns:
        dict: Dictionary mapping filename to corresponding DataFrame.
    """

    # Get list of files in directory that start with 'distance_'
    files = [f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f)) and f.startswith('distance_')]

    # Create a dictionary to store dataframes
    dashing_dfs = {}

    # Read each file into a dataframe, skipping first line
    for file in files:
        file_path = os.path.join(dir_path, file)
        try:
            # Skip first line and read tab-separated data
            df = pd.read_csv(file_path, sep='\t', header=None, skiprows=1)
            df.index = df.iloc[:, 0].str.replace('genomes_ncbi/', '')
            df = df.iloc[:, 1:]
            df.columns = df.index
            dashing_dfs[file] = df
            # print(f"Successfully read {file}")
        except Exception as e:
            print(f"Error reading {file}: {str(e)}")

    # Print summary of loaded dataframes
    print(f"\nLoaded {len(dashing_dfs)} files")
    for file, df in dashing_dfs.items():
        print(f"{file}: {df.shape}")
    return dashing_dfs

In [40]:
cyano_dir_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/cyanobacteriales/dashingfull_genomes_ncbi'
chlam_dir_path = '/scratch4/blangme2/sjhwang/ok_k_mer/data/chlamydiales/dashingfull_genomes_ncbi'

cyano_dashing_dfs = parse_dashing_dfs(cyano_dir_path)
chlam_dashing_dfs = parse_dashing_dfs(chlam_dir_path)

Error reading distance_k33.txt: No columns to parse from file

Loaded 28 files
distance_k10.txt: (585, 585)
distance_k11.txt: (585, 585)
distance_k12.txt: (585, 585)
distance_k13.txt: (585, 585)
distance_k14.txt: (585, 585)
distance_k15.txt: (585, 585)
distance_k16.txt: (585, 585)
distance_k17.txt: (585, 585)
distance_k18.txt: (585, 585)
distance_k19.txt: (585, 585)
distance_k20.txt: (585, 585)
distance_k21.txt: (585, 585)
distance_k22.txt: (585, 585)
distance_k23.txt: (585, 585)
distance_k24.txt: (585, 585)
distance_k25.txt: (585, 585)
distance_k26.txt: (585, 585)
distance_k27.txt: (585, 585)
distance_k28.txt: (585, 585)
distance_k29.txt: (585, 585)
distance_k30.txt: (585, 585)
distance_k31.txt: (585, 585)
distance_k32.txt: (585, 585)
distance_k5.txt: (585, 585)
distance_k6.txt: (585, 585)
distance_k7.txt: (585, 585)
distance_k8.txt: (585, 585)
distance_k9.txt: (585, 585)
Error reading distance_k33.txt: No columns to parse from file

Loaded 28 files
distance_k10.txt: (288, 288)
distan

In [ ]:
# spearman rank for k to true distance for all pairs

def spearman_corr_against_tree(dfs_dict, tree_dict):
    rows = []
    for fname, df in dfs_dict.items():
        # Derive k from filename when possible
        k_val = int(str(fname).replace('distance_k', '').replace('.txt', ''))
        names = list(df.index)
        x_dashing = []
        y_tree = []

        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                raw1, raw2 = names[i], names[j]
                g1 = raw1.replace('genomes_ncbi/', '').rstrip('.fa')
                g2 = raw2.replace('genomes_ncbi/', '').rstrip('.fa')
                td = tree_dict[g1][g2]
                dashing_dist = float(df.loc[raw1, raw2])
                x_dashing.append(dashing_dist)
                y_tree.append(float(td))

        corr, pval = spearmanr(x_dashing, y_tree)
        rows.append({
            'k': k_val,
            'spearman_corr': corr,
            'p_value': pval,
            'n_pairs': len(x_dashing)
        })

    return pd.DataFrame(rows)


In [55]:

# spearman rank for k to true distance for all pairs

def spearman_corr_against_tree(dfs_dict, tree_dict, sample_mask=[]):
    rows = []
    for fname, df in dfs_dict.items():
        # Derive k from filename when possible
        k_val = int(str(fname).replace('distance_k', '').replace('.txt', ''))
        names = list(df.index)
        x_dashing = []
        y_tree = []

        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                raw1, raw2 = names[i], names[j]
                g1 = raw1.replace('genomes_ncbi/', '').rstrip('.fa')
                g2 = raw2.replace('genomes_ncbi/', '').rstrip('.fa')
                # Only analyze samples not in sample_mask
                if (g1 in sample_mask) or (g2 in sample_mask):
                    continue
                td = tree_dict[g1][g2]
                dashing_dist = float(df.loc[raw1, raw2])
                x_dashing.append(dashing_dist)
                y_tree.append(float(td))

        corr, pval = spearmanr(x_dashing, y_tree)
        rows.append({
            'k': k_val,
            'spearman_corr': corr,
            'p_value': pval,
            'n_pairs': len(x_dashing)
        })

    return pd.DataFrame(rows)


In [ ]:

def get_random_samples(sample_names, sample_fraction=0.1, seed=None):
    if seed is not None:
        set_seed(seed)
    num_samples = max(1, int(len(sample_names) * sample_fraction))
    random_samples = sample(sample_names, num_samples)
    return random_samples


In [54]:
cyano_mask_samples = get_random_samples(cyano_sample_names, sample_fraction=0.1, seed=42)
chlam_mask_samples = get_random_samples(chlam_sample_names, sample_fraction=0.1, seed=42)
len(cyano_mask_samples), len(chlam_mask_samples)

(58, 28)

In [ ]:
chlam_corr_df = spearman_corr_against_tree(chlam_dashing_dfs, chlamydia_nwk_dict, chlam_mask_samples)

# Sort by numeric k when possible
if 'k' in chlam_corr_df.columns:
    chlam_corr_df = chlam_corr_df.sort_values('k')

chlam_corr_df

/tmp/ipykernel_2091548/2749167571.py:25: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.


,k,spearman_corr,p_value,n_pairs
23,5,NaN,NaN,33670
24,6,-0.060740,6.741745e-29,33670
25,7,-0.186053,6.495043e-260,33670
26,8,-0.329635,0.000000e+00,33670
27,9,-0.332804,0.000000e+00,33670
0,10,-0.348308,0.000000e+00,33670
1,11,-0.347016,0.000000e+00,33670
2,12,-0.295782,0.000000e+00,33670
3,13,-0.246704,0.000000e+00,33670
4,14,-0.229422,0.000000e+00,33670


In [ ]:
chlam_corr_df = spearman_corr_against_tree(chlam_dashing_dfs, chlamydia_nwk_dict)

# Sort by numeric k when possible
if 'k' in chlam_corr_df.columns:
    chlam_corr_df = chlam_corr_df.sort_values('k')

chlam_corr_df

/tmp/ipykernel_2091548/2749167571.py:25: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.


In [ ]:
# plot

In [ ]:
# Filter for just the distance_k files and sort them by k value
distance_files = sorted([f for f in dashing_dfs.keys() if f.startswith('distance_k')],
                       key=lambda x: int(x.replace('distance_k', '').replace('.txt', '')))

# Create list of pairs and their distances
pairs = []
k_values = {}

# Process first file to get genome names and create pairs
first_df = dashing_dfs[distance_files[0]]
genomes = first_df.iloc[0, 1:].tolist()  # Get genome names from first row

# Clean genome names by removing 'genomes_ncbi/'
genomes = [g.replace('genomes_ncbi/', '') for g in genomes]

# Create all pairs of genomes (upper triangle only)
for i in range(len(genomes)):
    for j in range(i+1, len(genomes)):
        # Sort lexicographically and join with '_'
        pair = '_'.join(sorted([genomes[i], genomes[j]]))
        pairs.append(pair)

# Initialize dataframe with pairs as index
merged_df = pd.DataFrame(index=pairs)

# Add data from each distance file
for file in distance_files:
    k = int(file.replace('distance_k', '').replace('.txt', ''))
    df = dashing_dfs[file]

    # Extract distances for each pair
    distances = []
    for i in range(len(genomes)):
        for j in range(i+1, len(genomes)):
            distances.append(df.iloc[i+1, j+1])

    # Add as new column
    merged_df[f'k_{k}'] = distances

# Clean up the index name
merged_df.index.name = 'genome_pair'


# Add metadat columns to merged_df
genome_pairs = []
genomes_1s = []
genomes_2s = []
tree_distances = []

for index in merged_df.index:
    # Split the genome pair
    genome1 = "_".join(index.split("_", 2)[:2])
    genome2 = "_".join(index.split("_", 2)[2:])

    # Get distance from tree_dict
    distance = tree_dict[genome1][genome2]

    genome_pairs.append(index)
    genomes_1s.append(genome1)
    genomes_2s.append(genome2)
    tree_distances.append(distance)

# Add tree_distance column to merged_df
merged_df['genome_pair'] = genome_pairs
merged_df['genome_1'] = genomes_1s
merged_df['genome_2'] = genomes_2s
merged_df['tree_distance'] = tree_distances

print(f"Final dataframe shape: {merged_df.shape}")
merged_df


# Backup, tmp

In [ ]:

# spearman rank for k to true distance for all pairs

def spearman_corr_against_tree(dfs_dict, tree_dict):
    rows = []
    for fname, df in dfs_dict.items():
        # Derive k from filename when possible
        try:
            k_val = int(str(fname).replace('distance_k', '').replace('.txt', ''))
        except Exception:
            k_val = fname

        names = list(df.index)
        x_dashing = []
        y_tree = []

        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                raw1, raw2 = names[i], names[j]
                g1 = raw1.replace('genomes_ncbi/', '').rstrip('.fa')
                g2 = raw2.replace('genomes_ncbi/', '').rstrip('.fa')

                # Fetch tree distance (symmetric)
                td = None
                if g1 in tree_dict and g2 in tree_dict[g1]:
                    td = tree_dict[g1][g2]
                elif g2 in tree_dict and g1 in tree_dict[g2]:
                    td = tree_dict[g2][g1]
                else:
                    continue  # skip pairs not present in the tree dict

                try:
                    dashing_dist = float(df.loc[raw1, raw2])
                except Exception:
                    # If labels aren't perfectly aligned, try the symmetric access
                    try:
                        dashing_dist = float(df.loc[raw2, raw1])
                    except Exception:
                        continue

                x_dashing.append(dashing_dist)
                y_tree.append(float(td))

        if len(x_dashing) >= 2:
            corr, pval = spearmanr(x_dashing, y_tree)
        else:
            corr, pval = np.nan, np.nan

        rows.append({
            'k': k_val,
            'spearman_corr': corr,
            'p_value': pval,
            'n_pairs': len(x_dashing)
        })

    return pd.DataFrame(rows)